# HW1：學生期中成績分析

## 開始前請先執行以下步驟

**1. 在終端機複製模板到 submit/ 並改名（將學號填入）：**
```bash
cp template.ipynb submit/HW1_<學號>.ipynb
```

**2. 開啟 `submit/HW1_<學號>.ipynb` 作答，請勿直接修改 `template.ipynb`。**

---

請在每個 `# TODO` 區塊填入你的程式碼。  
完成後執行最後一格，會自動在同一目錄產生 `HW1_<學號>.py`。

> ⚠️ **請勿修改函式名稱、參數與 return 結構**，否則自動評分將無法正常運作。

In [63]:
!pip install pandas numpy -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [64]:
import os
import pandas as pd

# 自動尋找 grades.csv（從 submit/ 或根目錄皆可）
_here = os.getcwd()
for _candidate in ['../grades.csv', 'grades.csv']:
    if os.path.exists(_candidate):
        GRADES_CSV = _candidate
        break


---
## 任務一：讀取與探索資料（不計分）

In [65]:
def load_and_explore_data(file_path):
    """任務一：讀取 CSV 並初步探索資料"""
    df = pd.read_csv(file_path, encoding='utf-8-sig')  # ← 請勿修改此行

    # TODO 1.1: 顯示前 5 筆資料
    print(df.head())

    # TODO 1.2: 查看資料結構（欄位、型態、缺失值）
    print(df.columns)
    print(df.dtypes)
    print(df.isnull().sum())


    return df  # ← 請勿修改 return

In [66]:
# 單題測試
df = load_and_explore_data(GRADES_CSV)

        姓名 性別 班級  數學  英文  國文  自然  社會
0    Alice  F  A  95  88  90  85  80
1      Bob  M  B  58  92  60  55  60
2  Charlie  M  A  77  85  80  70  90
3    David  M  B  45  70  55  60  50
4      Eva  F  A  88  95  92  89  85
Index(['姓名', '性別', '班級', '數學', '英文', '國文', '自然', '社會'], dtype='str')
姓名      str
性別      str
班級      str
數學    int64
英文    int64
國文    int64
自然    int64
社會    int64
dtype: object
姓名    0
性別    0
班級    0
數學    0
英文    0
國文    0
自然    0
社會    0
dtype: int64


---
## 任務二：新增衍生欄位

In [67]:
def feature_engineering(df):
    """任務二：計算總分、平均分數與是否及格"""

    subject = ["數學", "英文", "國文", "自然", "社會"]

    # TODO 2.1: 計算總分（五科加總）
    df['總分'] = df[subject].sum(axis = 1)

    # TODO 2.2: 計算平均分數
    df['平均'] = df[subject].mean(axis = 1)

    # TODO 2.3: 新增是否及格欄位（平均 >= 60 為及格）
    df['是否及格'] = df['平均'] >= 60

    return df  # ← 請勿修改 return

In [68]:
# 單題測試
df = feature_engineering(df.copy())
df.head()

,姓名,性別,班級,數學,英文,國文,自然,社會,總分,平均,是否及格
0,Alice,F,A,95,88,90,85,80,438,87.6,True
1,Bob,M,B,58,92,60,55,60,325,65.0,True
2,Charlie,M,A,77,85,80,70,90,402,80.4,True
3,David,M,B,45,70,55,60,50,280,56.0,False
4,Eva,F,A,88,95,92,89,85,449,89.8,True


---
## 任務三：資料篩選

In [69]:
def filter_and_analyze_data(df):
    """任務三與五：篩選資料與統計"""

    # TODO 3.1: 找出數學成績 < 60 的學生
    math_failed = df[df["數學"] < 60]

    # TODO 3.2: 找出班級為 'A' 且英文 > 90 的學生
    high_A = df[(df["班級"] == 'A') & (df["英文"] > 90)]

    # TODO 5.1: 顯示所有科目及平均分數的統計摘要
    summary = df[['國文', '英文', '數學', '自然', '社會', '平均']].describe()

    # TODO 5.2: 找出總分最高的學生
    # Hint: 可以先找到總分最高分，再篩選對應學生
    max_total = df['總分'].max()
    top_student = df[df['總分']  == max_total]

    return {  # ← 請勿修改 return 結構（key 名稱不可變動）
        "processed_df": df,
        "math_failed": math_failed,
        "high_A": high_A,
        "summary": summary,
        "top_student": top_student
    }

In [70]:
# 單題測試
result = filter_and_analyze_data(df.copy())
print("數學不及格學生：")
print(result['math_failed'])
print("\nA班英文>90學生：")
print(result['high_A'])
#print(result['top_student'])

數學不及格學生：
      姓名 性別 班級  數學  英文  國文  自然  社會   總分    平均   是否及格
1    Bob  M  B  58  92  60  55  60  325  65.0   True
3  David  M  B  45  70  55  60  50  280  56.0  False
8    Ivy  F  A  55  80  65  60  70  330  66.0   True

A班英文>90學生：
    姓名 性別 班級  數學  英文  國文  自然  社會   總分    平均  是否及格
4  Eva  F  A  88  95  92  89  85  449  89.8  True


---
## 任務四：分組統計（groupby）

In [71]:
def group_statistics(df):
    """任務四：使用 groupby 進行分組統計"""

    # TODO 4.1: 計算各班級的平均總分
    # Hint: df.groupby(...)['總分'].mean()
    class_avg_total = df.groupby('班級')['總分'].mean()

    # TODO 4.2: 計算各性別的及格率
    # Hint: 是否及格欄位為 True/False，mean() 可直接計算比例
    gender_pass_rate = df.groupby('性別')['是否及格'].mean()

    return {  # ← 請勿修改 return 結構（key 名稱不可變動）
        "class_avg_total": class_avg_total,
        "gender_pass_rate": gender_pass_rate
    }

In [72]:
# 單題測試
group_result = group_statistics(df.copy())
print("各班級平均總分：")
print(group_result['class_avg_total'])
print("\n各性別及格率：")
print(group_result['gender_pass_rate'])

各班級平均總分：
班級
A    404.750000
B    326.000000
C    397.333333
Name: 總分, dtype: float64

各性別及格率：
性別
F    1.000000
M    0.833333
Name: 是否及格, dtype: float64


---
## 任務五：敘述統計分析

（程式碼已在任務三的 `filter_and_analyze_data` 中完成，此格用來觀察結果）

In [73]:
# 單題測試
print("統計摘要：")
print(result['summary'])
print("\n總分最高的學生：")
print(result['top_student'])

統計摘要：
              國文         英文          數學         自然        社會         平均
count  10.000000  10.000000   10.000000  10.000000  10.00000  10.000000
mean   76.400000  81.800000   73.300000  72.900000  74.50000  75.780000
std    13.817863  11.679041   18.427335  14.441068  14.02577  13.136023
min    55.000000  65.000000   45.000000  55.000000  50.00000  56.000000
25%    66.250000  71.750000   58.500000  60.000000  62.50000  65.250000
50%    77.500000  82.500000   74.500000  75.000000  79.00000  77.500000
75%    88.000000  91.000000   86.750000  85.000000  83.75000  86.050000
max    95.000000  98.000000  100.000000  90.000000  92.00000  95.000000

總分最高的學生：
      姓名 性別 班級   數學  英文  國文  自然  社會   總分    平均  是否及格
6  Grace  F  C  100  98  95  90  92  475  95.0  True


---
## 任務六：儲存結果

In [74]:
def save_results(df, output_file_path):
    """任務六：儲存為 CSV"""

    # TODO 6.1: 儲存 CSV，避免中文亂碼
    # Hint: df.to_csv(...)
    df.to_csv(output_file_path, index=False, encoding='utf-8-sig')

In [75]:
# 單題測試
save_results(result['processed_df'], 'grades_analyzed.csv')
print("✅ 已儲存 grades_analyzed.csv")

✅ 已儲存 grades_analyzed.csv


---
## 繳交：自動產生 submit/HW1_<學號>.py

填入學號後執行此格，即可完成繳交準備。

In [76]:
import inspect

STUDENT_ID = "112502528"  # ← 修改為你的學號（9 位數字）

# ----------------------------------------------------------------
# 以下請勿修改
# ----------------------------------------------------------------
FUNCS = [
    load_and_explore_data,
    feature_engineering,
    filter_and_analyze_data,
    group_statistics,
    save_results,
]

MAIN_BLOCK = '''
if __name__ == "__main__":
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    INPUT_CSV = os.path.join(BASE_DIR, '..', 'grades.csv')
    OUTPUT_CSV = os.path.join(BASE_DIR, 'grades_analyzed.csv')

    df = load_and_explore_data(INPUT_CSV)
    df = feature_engineering(df)
    result = filter_and_analyze_data(df)
    group_result = group_statistics(result["processed_df"])
    save_results(result["processed_df"], OUTPUT_CSV)

    print("完成所有分析任務")
'''

lines = [
    '# -*- coding: utf-8 -*-',
    'import os',
    'import pandas as pd',
    '',
]
for func in FUNCS:
    lines.append(inspect.getsource(func))
    lines.append('')
lines.append(MAIN_BLOCK)

# 輸出到與此 notebook 相同的目錄（submit/）
output_path = f'HW1_{STUDENT_ID}.py'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print(f'✅ 已產生 {output_path}，請將此檔案提交 Pull Request')


✅ 已產生 HW1_112502528.py，請將此檔案提交 Pull Request
